# Explore SA-1B WebDataset images and masks

Browse the **stored, uncropped images** and independent instance masks in the prepared JPEG/JSON tar shards. Masks use COCO RLE at their **original resolution** (unresized); overlaps are preserved.

Install the environment from the repository root:
```bash
uv sync --extra data-prep --extra notebooks
uv run --extra data-prep --extra notebooks jupyter lab notebooks/explore_sa1b_webdataset.ipynb
```
For private Backblaze data, either export `B2_APPLICATION_ID` and `B2_APPLICATION_KEY` in the shell before launching Jupyter, or leave them unset and the next cell will prompt for them (hidden input, kept only in this kernel's environment). Credentials stay in the environment, not notebook cells or outputs. A local shard directory works without credentials.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents]
             if (path / "datasets/sa1b_webdataset.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter from the repository root or its notebooks directory")
sys.path.insert(0, str(ROOT))

import itertools
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
from datasets.sa1b_webdataset import (
    resolve_shards, iter_encoded_samples, decode_sample, decode_masks,
    sample_split, training_sample,
)


## Choose the source

`SOURCE` accepts a local directory/glob, an explicit list of shard URLs, or a private `s3://bucket/prefix/`. Only completed `.tar` files are listed. This snapshots the currently available shards; rerun the cell to discover new uploads.

Reading does not download entire shards to disk. Previewing a late position still requires streaming past earlier samples in that shard.


In [ ]:
import getpass

# Only prompts if not already exported in the shell that launched Jupyter.
# Values are entered hidden and kept in-process (os.environ), never echoed
# into a cell or its output.
# if not os.environ.get("B2_APPLICATION_ID"):
os.environ["B2_APPLICATION_ID"] = getpass.getpass("B2_APPLICATION_ID: ")
# if not os.environ.get("B2_APPLICATION_KEY"):
os.environ["B2_APPLICATION_KEY"] = getpass.getpass("B2_APPLICATION_KEY: ")


In [ ]:
SOURCE = os.environ.get("SA1B_WDS_SOURCE", "s3://sa1b-webdataset/sa1b-896/")
ENDPOINT = os.environ.get("B2_ENDPOINT_URL", "https://s3.eu-central-003.backblazeb2.com")
# Local alternative:
# SOURCE = str(ROOT / "datasets/sa1b-webdataset/shards")

SHARDS = resolve_shards(SOURCE, ENDPOINT)
print(f"Found {len(SHARDS):,} completed shards")


## Load a small preview

Choose a shard and a starting offset. Only this preview's images and JSON are held in memory. Rerun this cell to fetch another group; rerun the viewer cell afterward to refresh its controls.


In [ ]:
SHARD_INDEX = 0
START_AT = 0
PREVIEW_COUNT = 6

if not 0 <= SHARD_INDEX < len(SHARDS):
    raise ValueError("SHARD_INDEX is out of range")
if START_AT < 0 or PREVIEW_COUNT < 1:
    raise ValueError("Use START_AT >= 0 and PREVIEW_COUNT >= 1")

preview = []
stream = iter_encoded_samples(SHARDS[SHARD_INDEX], ENDPOINT)
try:
    for sample in itertools.islice(stream, START_AT, START_AT + PREVIEW_COUNT):
        image, metadata = decode_sample(sample)
        preview.append({"key": sample["__key__"], "image": image, "metadata": metadata,
                        "encoded": sample})
finally:
    stream.close()
if not preview:
    raise ValueError("No samples at this offset; reduce START_AT or choose another shard")

for item in preview:
    print(f"{item['key']}: {item['image'].width} × {item['image'].height}, "
          f"{len(item['metadata']['annotations'])} masks, "
          f"split={sample_split(item['key'])}")


## Image, mask overlay, and one instance

Select an image and a mask. The overlay displays the first selected number of masks; the right panel isolates one annotation. Colors are consistent within each image. All masks are independent, so a pixel may belong to several regions. The image is shown at its stored dimensions, before training resize/crop/normalization.

Masks are stored at their **original** resolution (not resized to match the stored image); they are decoded against `metadata["original_size"]` and resized down with nearest-neighbor only here, for this overlay's display.


In [ ]:
image_selector = widgets.Dropdown(options=[(item["key"], i) for i, item in enumerate(preview)], description="Image")
mask_selector = widgets.IntSlider(value=0, min=0, max=0, description="Mask", continuous_update=False)
alpha_control = widgets.FloatSlider(value=0.45, min=0, max=1, step=0.05, description="Opacity", continuous_update=False)
overlay_count = widgets.IntSlider(value=50, min=1, max=150, description="Overlay count", continuous_update=False)
viewer = widgets.Output()


def render_preview(*_):
    item = preview[image_selector.value]
    annotations = item["metadata"]["annotations"]
    count = len(annotations)
    mask_index = min(mask_selector.value, max(0, count - 1))
    base = np.asarray(item["image"], dtype=np.float32) / 255
    overlay = base.copy()
    chosen = np.zeros(base.shape[:2], dtype=bool)
    colors = np.random.default_rng(42).uniform(0.15, 1, (max(1, count), 3))
    mask_height, mask_width = item["metadata"]["original_size"]
    for i, mask in enumerate(decode_masks(item["metadata"], (mask_width, mask_height))):
        if mask.shape != base.shape[:2]:
            # Masks are stored at original resolution; resize only for display here.
            mask = np.asarray(Image.fromarray(mask).resize(item["image"].size, Image.Resampling.NEAREST))
        region = mask.astype(bool)
        if i < overlay_count.value:
            overlay[region] = (1 - alpha_control.value) * overlay[region] + alpha_control.value * colors[i]
        if i == mask_index:
            chosen = region
        if i >= max(mask_index, overlay_count.value - 1):
            break
    with viewer:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 3, figsize=(16, 6))
        axes[0].imshow(base)
        axes[0].set_title(f"{item['key']} — stored image")
        axes[1].imshow(overlay)
        axes[1].set_title(f"Overlay: {min(count, overlay_count.value)} / {count} masks")
        axes[2].imshow(chosen, cmap="gray", vmin=0, vmax=1)
        if count:
            annotation = annotations[mask_index]
            axes[2].set_title(f"Mask {mask_index} · annotation {annotation['id']}\n{int(chosen.sum()):,} pixels")
        else:
            axes[2].set_title("No masks in this sample")
        for axis in axes:
            axis.axis("off")
        plt.tight_layout()
        plt.show()
        plt.close(fig)


def change_image(*_):
    count = len(preview[image_selector.value]["metadata"]["annotations"])
    mask_selector.max = max(0, count - 1)
    mask_selector.value = 0
    mask_selector.disabled = count == 0
    render_preview()

image_selector.observe(change_image, names="value")
for control in (mask_selector, alpha_control, overlay_count):
    control.observe(render_preview, names="value")
display(widgets.VBox([image_selector, widgets.HBox([mask_selector, alpha_control]), overlay_count, viewer]))
change_image()


## Inspect the training tensors

This uses the same deterministic resize and center crop as training, with bilinear images and nearest-neighbor masks. Set `TRAIN_SIZE=224` for stage 1 or `896` for stage 2. Tensor conversion below is **before image normalization** for easy visualization; the training scripts add their existing backbone normalization.

The labels have shape `[150, H, W]`, contain `0/1` for real masks, and `-1` for unused slots. The first 150 annotations are retained. Unlike storage preprocessing, training can upscale a smaller image to its requested input size.


In [ ]:
import torch
from torchvision import transforms as T

TRAIN_SIZE = 224
image_transform = T.Compose([
    T.Resize(TRAIN_SIZE, T.InterpolationMode.BILINEAR),
    T.CenterCrop(TRAIN_SIZE), T.ToTensor(),
])
mask_transform = T.Compose([
    T.PILToTensor(), T.Resize(TRAIN_SIZE, T.InterpolationMode.NEAREST),
    T.CenterCrop(TRAIN_SIZE),
])
training = training_sample(preview[image_selector.value]["encoded"], image_transform, mask_transform, max_masks=150)
valid = (training["label"] >= 0).flatten(1).any(1)
print("Image:", tuple(training["img"].shape), training["img"].dtype)
print("Masks:", tuple(training["label"].shape), training["label"].dtype)
print("Real mask slots:", int(valid.sum()), "| padded:", int((~valid).sum()))
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(training["img"].permute(1, 2, 0))
axes[0].set_title("Training image (before normalization)")
axes[1].imshow((training["label"] > 0).any(0), cmap="gray")
axes[1].set_title("Union of training masks")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()
plt.close(fig)


## Train from these shards

From the repository root, with B2 credentials exported:
```bash
uv run --extra data-prep python train_loftup_stage1.py \
  dataset=sa1b_webdataset \
  webdataset.shards=s3://sa1b-webdataset/sa1b-896/ \
  webdataset.endpoint=https://s3.eu-central-003.backblazeb2.com \
  webdataset.train_batches=1000 webdataset.val_batches=100
```
The same options work for stage 2 with its required pretrained checkpoint. Epoch lengths are **per-rank batch budgets**, not an exhaustive dataset pass. Streams repeat if needed; validation uses deterministic order. Both stages must use the same split seed/fraction. This hash split differs from the older flat-file loader's shuffled split. Keep the shard set fixed during a run.
